In [1]:
from deepinv.loss.metric import PSNR, LPIPS
from deepinv.physics import Decolorize, Blur, BlurFFT
from deepinv.models import GSDRUNet
from deepinv.physics.blur import gaussian_blur
import torch
import numpy as np
import os
from priors import GSDPrior, L2
from prior_comparison import DegradedLikelihood
from utils import device, dict_to_json, laplace, moffat, uniform
from experiments_utils import generate_blur_operator
from sampling import SKROCK
import sys
from torchvision.datasets import ImageFolder
from torchvision.transforms import Resize
from torchvision import transforms
import matplotlib.pyplot as plt


kernels = [gaussian_blur(sigma=(2, 2)),
           moffat((0.5, 1), size=7),
           laplace(0.4, size=10), uniform(3), 
           gaussian_blur(sigma=(2.5, 2.5)),]

physics = [Blur(img_size=(1, 256, 256), filter=kernels[i], 
                       device=device, padding='valid').to(device) for i in range(5)]
physics_full = [BlurFFT(img_size=(1, 256, 256), filter=kernels[i], 
                       device=device).to(device) for i in range(5)]
nkernels = len(kernels)

res_mat = np.zeros([5, 5, 3])
res_mat_perc = np.zeros([5, 5, 3])

# determining max padding for the FFT blur
cuts = np.zeros(5, dtype=int)
x = torch.zeros(1, 1, 256, 256).to(device)
for i in range(5):
    cuts[i] = (256 - physics[i](x).shape[-1])//2
cut = np.max(cuts)

In [2]:
cut_padding = True
metrics_x = dict()
for met in ['euclidian', 'euclidianGT']:
    metrics_x[met] = np.zeros([5, 5, 3])
for i in range(5):  # GT kernel
    physics_GT = physics_full[i]
    for j in range(5):
        physics_loc = physics_full[j]
        for k in range(3):
            trace_dict = np.load("results/kernel_comparison/trace_{}_{}_{}.npz".format(i, j, k))
            x = torch.tensor(trace_dict["x"], device=device)
            x_samples = torch.tensor(trace_dict["samples_x"], device=device).clamp(0, 1)
            nb_noise, nb_steps = x_samples.shape[0], x_samples.shape[1]
            x_samples = x_samples.reshape((-1,) + x.shape[-3:])
            y = trace_dict["y"]
            yp_samples = torch.tensor(trace_dict["samples_yp"], device=device)
            x_samples_proj = physics_loc(x_samples.view((-1,) + x_samples.shape[-3:]))
            x_proj = physics_GT(x.view((-1,) + x_samples.shape[-3:]))
            if cut_padding:
                x = x[:, :, cut:256-cut, cut:256-cut]
                x_samples_proj = x_samples_proj[:, :, cut:256-cut, cut:256-cut]
                x_samples = x_samples[:, :, cut:256-cut, cut:256-cut]
                y = y[:, :, cut:256-cut, cut:256-cut]
                yp_samples = yp_samples[:, :, cut:256-cut, cut:256-cut]
                x_proj= x_proj[:, :, cut:256-cut, cut:256-cut]
            # reshape so first dim correspond to noise it, (nb_noise, nb_steps, c, h, w)
            x_samples = x_samples.reshape((nb_noise, nb_steps) + x.shape[-3:])
            yp_samples = yp_samples.reshape((nb_noise, -1))
            x_samples_proj = x_samples_proj.reshape((nb_noise, nb_steps) + y.shape[-3:])
            for t in range(nb_noise):
                metrics_x["euclidian"][i, j, k] +=  torch.sum(torch.norm(yp_samples[t].view([1, -1]) - x_samples_proj[t].reshape([nb_steps, -1]), dim=-1)**2)      
                metrics_x["euclidianGT"][i, j, k] +=  torch.sum(torch.norm(x_proj.reshape([1, -1]) - x_samples_proj[t].reshape([nb_steps, -1]), dim=-1)**2)      
                
            metrics_x["euclidianGT"][i, j, k] = metrics_x["euclidianGT"][i, j, k]/nb_steps/nb_noise    
            metrics_x["euclidian"][i, j, k] = metrics_x["euclidian"][i, j, k]/nb_steps/nb_noise    

In [3]:
acc = 0
amin = np.argmin(metrics_x["euclidian"], axis=1)
for i in range(5):
        for t in range(3):
            if amin[i, t] == i:
                acc += 1
print("single shot accuracy: {} %".format(acc/15*100))

single shot accuracy: 86.66666666666667 %


In [4]:
mean_amin = np.argmin(np.mean(metrics_x["euclidian"]**2, axis=2), axis=1)
print("Few shots accuracy: {} %".format(np.mean(mean_amin == np.arange(5))*100))

Few shots accuracy: 100.0 %
